# GEPA without textual feedback comparison code

In [ ]:
# ============================================================
# 0) IMPORTS & CONFIG
# ============================================================
import dspy
import torch
import re
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from svlearn_dspy.llm import make_lm

# LLM_PROVIDER in .env: supportvectors -> SV cluster, else hosted OpenAI.
# Task model is gpt-5-mini (or gpt-oss-20b); reflection is gpt-5 (or gpt-oss-120b).
lm = make_lm()
dspy.configure(lm=lm)

reflection_lm = make_lm(strong=True)


In [ ]:
dataset = load_dataset("mlabonne/smoltldr")

examples = [
    dspy.Example({"prompt": x["prompt"], "gold_tldr": x["completion"]}).with_inputs("prompt")
    for x in dataset["train"]
]

total = len(examples)  # should be ~2000

train_end = int(0.80 * total)   # 80%
val_end   = int(0.90 * total)   # next 10%

train_set = examples[:train_end]
val_set   = examples[train_end:val_end]
test_set  = examples[val_end:]

print(len(train_set), len(val_set), len(test_set))


In [ ]:
# ============================================================
# 2) SIGNATURE + MODULE
# ============================================================
class GenerateTLDR(dspy.Signature):
    """
    Given a Reddit post that begins with 'POST:', generate a single-line,
    concise TL;DR (~25 words). Avoid line breaks.
    """
    prompt: str = dspy.InputField()
    tldr: str = dspy.OutputField()


class TLDRModule(dspy.Module):
    def __init__(self):
        self.generator = dspy.ChainOfThought(GenerateTLDR)

    def forward(self, prompt: str):
        out = self.generator(prompt=prompt)
        return dspy.Prediction(tldr=out.tldr)

program = TLDRModule()


In [ ]:
# ============================================================
# 3) HELPERS (semantic model, post text extractor)
# ============================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # avoid HF tokenizer thread issues

# Load the semantic model ONCE, globally — no lazy loading in threads
sem_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def _word_count(s: str) -> int:
    return len(s.split())

def extract_post(prompt: str):
    m = re.search(r"POST:\s*(.*?)(?:\n\s*TL;DR:|$)", prompt, flags=re.DOTALL)
    return m.group(1).strip() if m else prompt.strip()



In [ ]:
# ============================================================
# 4) FEEDBACK FUNCTIONS — EXACTLY LIKE YOUR STYLE
# ============================================================

def feedback_len(pred_tldr: str, target_words=25, max_words=70):
    """
    Length feedback (normalized 0–1 score).
    """
    n = _word_count(pred_tldr)
    max_diff = max(abs(target_words - 1), abs(target_words - max_words))
    score = 1 - (abs(target_words - n) / max_diff)
    score = max(0.0, min(1.0, score))

    if score == 1.0:
        fb = f"You produced the ideal length (~{n} words)."
    elif n > target_words:
        fb = f"Your TL;DR is too long ({n} words). Aim for ~{target_words}."
    else:
        fb = f"Your TL;DR is too short ({n} words). Add key content."
    return fb, score


def feedback_style(pred_tldr: str):
    """
    Style feedback (1 if single line, else 0).
    """
    ok = ("\n" not in pred_tldr) and ("\r" not in pred_tldr)
    score = 1.0 if ok else 0.0
    if score == 1.0:
        fb = "You kept it single-line with no line breaks."
    else:
        fb = "Your TL;DR contains line breaks. It must be a single line."
    return fb, score


def feedback_sem(prompt: str, pred_tldr: str):
    """
    Semantic similarity feedback (normalized 0–1).
    """
    post = extract_post(prompt)

    # Use the globally loaded sem_model (no lazy creation in threads)
    global sem_model
    with torch.no_grad():
        e_pred = sem_model.encode(pred_tldr, convert_to_tensor=True, normalize_embeddings=True)
        e_post = sem_model.encode(post,      convert_to_tensor=True, normalize_embeddings=True)
        sim = torch.sum(e_pred * e_post).item()

    score = (sim + 1) / 2        # [-1,1] -> [0,1]
    score = max(0.0, min(1.0, score))

    if score > 0.8:
        fb = "Excellent semantic alignment — you captured the core meaning."
    elif score > 0.6:
        fb = "Moderate alignment — include 1–2 more key ideas."
    else:
        fb = "Low alignment — your TL;DR misses core points of the post."
    return fb, score

In [ ]:

# ============================================================
# 5) METRIC (scalar only)
# ============================================================
def metric(example, pred, trace=None, pred_name=None, pred_trace=None):
    fb_len, s_len = feedback_len(pred.tldr)
    fb_sty, s_sty = feedback_style(pred.tldr)
    fb_sem, s_sem = feedback_sem(example["prompt"], pred.tldr)

    total = (s_len + s_sty + s_sem) / 3.0
    return total


In [ ]:
# ============================================================
# BASELINE EVALUATION
# ============================================================
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=16,
    display_table=True,
    display_progress=True
)

print("\n== Baseline Evaluation ==")
evaluate(program)

In [ ]:

# ============================================================
# 7) GEPA OPTIMIZER
# ============================================================
from dspy import GEPA

optimizer = GEPA(
    metric=metric,
    auto="light",
    num_threads=16,
    track_stats=True,
    track_best_outputs=True,
    use_merge=False,
    reflection_lm=reflection_lm
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)


In [ ]:
# ============================================================
# 8) SEE OPTIMIZED PROMPTS
# ============================================================
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")


In [ ]:
# ============================================================
# 9) FINAL POST-GEPA EVALUATION
# ============================================================
print("\n== GEPA-Optimized Evaluation ==")
evaluate(optimized_program)

## Performance Improvement Summary

Display the accuracy lift from baseline (90.92%) to optimized (93.21%).

### Comparison: GEPA vs MIPROv2

| Optimizer | Baseline | Optimized | Improvement |
|-----------|----------|-----------|-------------|
| GEPA | 90.92% | 93.21% | +2.29 pp |
| MIPROv2 | 90.92% | 91.13% | +0.21 pp |
| GEPA W/O FEEDBACK| 90.7% | 91.83%| +1.13pp |

*pp = percentage points*  
Corresponding MIPROv2 results are documented in `07_miprov2_3_tldr.ipynb`. 
